In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Load train and test datasets
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print(train.shape, test.shape)
train.head()


In [ ]:
# Basic info
print(train.info())

# Check missing values
train.isnull().sum().sort_values(ascending=False).head(20)

# Distribution of target
import matplotlib.pyplot as plt
import seaborn as sns

sns.histplot(train["SalePrice"], kde=True)
plt.show()


In [ ]:
# Log transform target
train["SalePrice"] = np.log1p(train["SalePrice"])

# Combine datasets for consistent preprocessing
train_id = train["Id"]
test_id = test["Id"]

y = train["SalePrice"]
X = train.drop(["SalePrice", "Id"], axis=1)
X_test = test.drop("Id", axis=1)

# Simple imputation for missing values
X = X.fillna(X.median(numeric_only=True))
X_test = X_test.fillna(X_test.median(numeric_only=True))

# One-hot encoding for categoricals
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)

# Align columns (make sure train and test match)
X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
}

results = {}
for name, model in models.items():
    scores = -cross_val_score(model, X, y, cv=5, scoring="neg_root_mean_squared_error")
    results[name] = scores.mean()

pd.DataFrame(results, index=["RMSE"])


In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns)
top10 = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=top10, y=top10.index)
plt.title("Top 10 Feature Importances")
plt.show()


In [ ]:
best_model = GradientBoostingRegressor(n_estimators=200, random_state=42)
best_model.fit(X, y)

preds = best_model.predict(X_test)
preds = np.expm1(preds)  # undo log transform

submission = pd.DataFrame({"Id": test_id, "SalePrice": preds})
submission.to_csv("submission.csv", index=False)
